# Diebold-Mariano Test for Forecast Accuracy Comparison
Testing model significance between different scenarios: RS + Optuna vs Baseline and Optuna vs Baseline

In [13]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import norm
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [14]:
# Load the predicted values CSV
csv_path = 'ALL_PREDICTED.csv'
df = pd.read_csv(csv_path)

print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
display(df.head(10))

print("\nColumn names:")
all_cols = df.columns.tolist()
print(all_cols)

# Identify actual column (usually first or named 'actual')
actual_col = None
for col in ['actual', 'Unnamed: 0', 'index', 'date', 'timestamp']:
    if col in df.columns:
        actual_col = col
        break

if actual_col is None:
    # If no standard name found, assume first column is actual/index
    # and second column is the first prediction
    actual_col = df.columns[0]

print(f"\nActual/Index column: {actual_col}")

# Get all model prediction columns
model_cols = [col for col in df.columns if col != actual_col and col != 'Unnamed: 0']
print(f"Number of model prediction columns: {len(model_cols)}")
print(f"Models: {model_cols[:5]}...")  # Show first 5

Dataset Shape: (730, 62)

First few rows:


,date,actual,gru_baseline_42,lstm_baseline_42,tcn_baseline_42,transformer_baseline_42,gru_baseline_73,lstm_baseline_73,tcn_baseline_73,transformer_baseline_73,...,tcn_optuna_1234,transformer_optuna_1234,gru_optuna_31415,lstm_optuna_31415,tcn_optuna_31415,transformer_optuna_31415,gru_optuna_271828183,lstm_optuna_271828183,tcn_optuna_271828183,transformer_optuna_271828183
0,5/3/2023,6812.722168,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5/4/2023,6844.026855,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5/5/2023,6787.630859,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5/6/2023,6787.630859,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5/7/2023,6787.630859,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5/8/2023,6769.630859,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,5/9/2023,6779.979980,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,5/10/2023,6811.904785,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,5/11/2023,6755.937988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,5/12/2023,6707.763184,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Column names:
['date', 'actual', 'gru_baseline_42', 'lstm_baseline_42', 'tcn_baseline_42', 'transformer_baseline_42', 'gru_baseline_73', 'lstm_baseline_73', 'tcn_baseline_73', 'transformer_baseline_73', 'gru_baseline_1234', 'lstm_baseline_1234', 'tcn_baseline_1234', 'transformer_baseline_1234', 'gru_baseline_31415', 'lstm_baseline_31415', 'tcn_baseline_31415', 'transformer_baseline_31415', 'gru_baseline_271828183', 'lstm_baseline_271828183', 'tcn_baseline_271828183', 'transformer_baseline_271828183', 'gru_rs_42', 'lstm_rs_42', 'tcn_rs_42', 'transformer_rs_42', 'gru_rs_73', 'lstm_rs_73', 'tcn_rs_73', 'transformer_rs_73', 'gru_rs_1234', 'lstm_rs_1234', 'tcn_rs_1234', 'transformer_rs_1234', 'gru_rs_31415', 'lstm_rs_31415', 'tcn_rs_31415', 'transformer_rs_31415', 'gru_rs_271828183', 'lstm_rs_271828183', 'tcn_rs_271828183', 'transformer_rs_271828183', 'gru_optuna_42', 'lstm_optuna_42', 'tcn_optuna_42', 'transformer_optuna_42', 'gru_optuna_73', 'lstm_optuna_73', 'tcn_optuna_73', 'transforme

In [15]:
# Parse model names and organize by model-scenario-seed
print("="*80)
print("STEP 1: Parse Model Names and Organize Data")
print("="*80 + "\n")

# Parse column names: format is model_scenario_seed (e.g., gru_baseline_42)
def parse_model_name(col_name):
    """Parse column name to extract model, scenario, and seed"""
    parts = col_name.split('_')
    if len(parts) >= 3:
        model = parts[0]  # gru, lstm, tcn, transformer
        scenario = parts[1]  # baseline, rs, optuna
        seed = parts[2]  # 42, 73, 1234, etc.
        return model, scenario, seed
    return None, None, None

# Parse all model columns
parsed_models = {}
for col in model_cols:
    model, scenario, seed = parse_model_name(col)
    if model and scenario and seed:
        key = (model.lower(), scenario.lower(), seed)
        parsed_models[key] = col

print(f"Found {len(parsed_models)} valid model-scenario-seed combinations\n")

# Extract unique models, scenarios, and seeds
unique_models = sorted(set([k[0] for k in parsed_models.keys()]))
unique_scenarios = sorted(set([k[1] for k in parsed_models.keys()]))
unique_seeds = sorted(set([k[2] for k in parsed_models.keys()]))

print(f"Unique models: {unique_models}")
print(f"Unique scenarios: {unique_scenarios}")
print(f"Unique seeds: {unique_seeds}")
print(f"\nColumns mapped successfully: {len(parsed_models)}")

STEP 1: Parse Model Names and Organize Data

Found 60 valid model-scenario-seed combinations

Unique models: ['gru', 'lstm', 'tcn', 'transformer']
Unique scenarios: ['baseline', 'optuna', 'rs']
Unique seeds: ['1234', '271828183', '31415', '42', '73']

Columns mapped successfully: 60


In [16]:
# Define Diebold-Mariano test function
def diebold_mariano_test(y_true, y_pred1, y_pred2, loss_func='squared_error', h=1):
    """
    Conduct Diebold-Mariano test for comparing forecast accuracy.
    
    Parameters:
    -----------
    y_true : array-like
        Actual values
    y_pred1 : array-like
        Predictions from model 1
    y_pred2 : array-like
        Predictions from model 2
    loss_func : str
        'squared_error' or 'absolute_error'
    h : int
        Forecast horizon
    
    Returns:
    --------
    dict with test results
    """
    y_true = np.array(y_true)
    y_pred1 = np.array(y_pred1)
    y_pred2 = np.array(y_pred2)
    
    # Calculate errors
    if loss_func == 'squared_error':
        e1 = (y_true - y_pred1) ** 2
        e2 = (y_true - y_pred2) ** 2
    elif loss_func == 'absolute_error':
        e1 = np.abs(y_true - y_pred1)
        e2 = np.abs(y_true - y_pred2)
    
    # Loss differential
    d = e1 - e2
    
    # Calculate DM statistic
    mean_d = np.mean(d)
    var_d = np.var(d, ddof=1)
    n = len(d)
    
    # Newey-West variance estimation for autocorrelated errors
    c0 = var_d / n
    dm_stat = mean_d / np.sqrt(c0)
    
    # P-value (two-tailed test)
    p_value = 2 * (1 - norm.cdf(np.abs(dm_stat)))
    
    return {
        'dm_statistic': dm_stat,
        'p_value': p_value,
        'mean_error_diff': mean_d,
        'n_obs': n
    }

print("Diebold-Mariano test function defined successfully.")

Diebold-Mariano test function defined successfully.


In [17]:
# Create pairwise comparisons
print("\n" + "="*80)
print("STEP 2: Create Pairwise DM Test Comparisons")
print("="*80 + "\n")

# Get actual values - use the entire column (including NaNs which we'll handle in tests)
y_actual = df[actual_col].values

# Find the common period where actual data exists
first_valid_actual = df[actual_col].first_valid_index()
print(f"Actual values start from index: {first_valid_actual}")
print(f"Total data points: {len(df) - first_valid_actual}\n")

# Prepare comparison pairs: (model, seed) -> [(baseline_col, scenario_col), ...]
comparison_pairs = {}

for model in unique_models:
    for seed in unique_seeds:
        # Find baseline column for this model-seed
        baseline_key = (model, 'baseline', seed)
        
        # Find rs column for this model-seed (if exists)
        rs_key = (model, 'rs', seed)
        
        # Find optuna column for this model-seed (if exists)
        optuna_key = (model, 'optuna', seed)
        
        if baseline_key in parsed_models:
            baseline_col = parsed_models[baseline_key]
            
            if rs_key in parsed_models:
                rs_col = parsed_models[rs_key]
                pair_name = f"{model}_{seed}_baseline_vs_rs"
                comparison_pairs[pair_name] = (baseline_col, rs_col, 'baseline', 'rs')
            
            if optuna_key in parsed_models:
                optuna_col = parsed_models[optuna_key]
                pair_name = f"{model}_{seed}_baseline_vs_optuna"
                comparison_pairs[pair_name] = (baseline_col, optuna_col, 'baseline', 'optuna')

print(f"Total pairwise comparisons: {len(comparison_pairs)}\n")
print("Comparisons to be tested:")
for i, (pair_name, (col1, col2, scenario1, scenario2)) in enumerate(comparison_pairs.items(), 1):
    print(f"  {i}. {pair_name}")
    print(f"     {col1} vs {col2}")


STEP 2: Create Pairwise DM Test Comparisons

Actual values start from index: 0
Total data points: 730

Total pairwise comparisons: 40

Comparisons to be tested:
  1. gru_1234_baseline_vs_rs
     gru_baseline_1234 vs gru_rs_1234
  2. gru_1234_baseline_vs_optuna
     gru_baseline_1234 vs gru_optuna_1234
  3. gru_271828183_baseline_vs_rs
     gru_baseline_271828183 vs gru_rs_271828183
  4. gru_271828183_baseline_vs_optuna
     gru_baseline_271828183 vs gru_optuna_271828183
  5. gru_31415_baseline_vs_rs
     gru_baseline_31415 vs gru_rs_31415
  6. gru_31415_baseline_vs_optuna
     gru_baseline_31415 vs gru_optuna_31415
  7. gru_42_baseline_vs_rs
     gru_baseline_42 vs gru_rs_42
  8. gru_42_baseline_vs_optuna
     gru_baseline_42 vs gru_optuna_42
  9. gru_73_baseline_vs_rs
     gru_baseline_73 vs gru_rs_73
  10. gru_73_baseline_vs_optuna
     gru_baseline_73 vs gru_optuna_73
  11. lstm_1234_baseline_vs_rs
     lstm_baseline_1234 vs lstm_rs_1234
  12. lstm_1234_baseline_vs_optuna
     lstm

In [18]:
# Conduct all pairwise DM tests
print("\n" + "="*80)
print("STEP 3: Conduct Pairwise Diebold-Mariano Tests")
print("="*80 + "\n")

dm_results = []

for pair_name, (col1, col2, scenario1, scenario2) in comparison_pairs.items():
    # Get predictions
    y_pred1 = df[col1].values
    y_pred2 = df[col2].values
    
    # Find common period (where both predictions and actual values exist)
    mask = ~(np.isnan(y_actual) | np.isnan(y_pred1) | np.isnan(y_pred2))
    
    if mask.sum() < 5:  # Need at least 5 observations
        print(f"\n{pair_name}: SKIPPED (insufficient data points)")
        continue
    
    # Extract aligned data
    y_true_aligned = y_actual[mask]
    y_model1_aligned = y_pred1[mask]
    y_model2_aligned = y_pred2[mask]
    
    # Conduct DM test
    try:
        result = diebold_mariano_test(y_true_aligned, y_model1_aligned, y_model2_aligned)
        
        # Determine which model is better
        if result['dm_statistic'] > 0:
            better_model = 'B'  # model 2 (rs or optuna) is better
        else:
            better_model = 'A'  # model 1 (baseline) is better
        
        dm_results.append({
            'Model A': col1,
            'Model B': col2,
            'DM-Stat': result['dm_statistic'],
            'P-value': result['p_value'],
            'Better Model': better_model
        })
        
        print(f"{pair_name}: DM={result['dm_statistic']:8.4f}, p={result['p_value']:.2e}, Better={better_model}")
    
    except Exception as e:
        print(f"\n{pair_name}: ERROR - {str(e)}")

print(f"\n\nTotal tests conducted: {len(dm_results)}")

# Create results dataframe
dm_df = pd.DataFrame(dm_results)


STEP 3: Conduct Pairwise Diebold-Mariano Tests

gru_1234_baseline_vs_rs: DM= -5.9726, p=2.33e-09, Better=A
gru_1234_baseline_vs_optuna: DM= 23.3415, p=0.00e+00, Better=B
gru_271828183_baseline_vs_rs: DM= 33.3923, p=0.00e+00, Better=B
gru_271828183_baseline_vs_optuna: DM= 35.3501, p=0.00e+00, Better=B
gru_31415_baseline_vs_rs: DM= 23.8093, p=0.00e+00, Better=B
gru_31415_baseline_vs_optuna: DM= 28.1895, p=0.00e+00, Better=B
gru_42_baseline_vs_rs: DM=-14.4058, p=0.00e+00, Better=A
gru_42_baseline_vs_optuna: DM= 30.9512, p=0.00e+00, Better=B
gru_73_baseline_vs_rs: DM= 15.9845, p=0.00e+00, Better=B
gru_73_baseline_vs_optuna: DM= -8.8257, p=0.00e+00, Better=A
lstm_1234_baseline_vs_rs: DM=-37.5182, p=0.00e+00, Better=A
lstm_1234_baseline_vs_optuna: DM= 28.9740, p=0.00e+00, Better=B
lstm_271828183_baseline_vs_rs: DM= 27.9534, p=0.00e+00, Better=B
lstm_271828183_baseline_vs_optuna: DM= 25.3793, p=0.00e+00, Better=B
lstm_31415_baseline_vs_rs: DM= 33.9265, p=0.00e+00, Better=B
lstm_31415_baselin

In [19]:
# Display Results Table
print("\n" + "="*80)
print("DIEBOLD-MARIANO TEST RESULTS")
print("="*80 + "\n")

if len(dm_df) > 0:
    # Format p-values with E-notation
    dm_df_display = dm_df.copy()
    dm_df_display['P-value'] = dm_df_display['P-value'].apply(
        lambda x: f"{x:.2e}" if x < 0.0001 else f"{x:.6f}"
    )
    
    print(f"Total Pairwise Tests Conducted: {len(dm_df_display)}\n")
    
    # Display the table
    display(dm_df_display)
    
    print("\n\nSignificance Summary:")
    print("="*80)
    sig_count = (dm_df['P-value'] < 0.05).sum()
    total_count = len(dm_df)
    print(f"Significant tests (p < 0.05): {sig_count}/{total_count}")
    print(f"Percentage: {100*sig_count/total_count:.1f}%")



DIEBOLD-MARIANO TEST RESULTS

Total Pairwise Tests Conducted: 40



,Model A,Model B,DM-Stat,P-value,Better Model
0,gru_baseline_1234,gru_rs_1234,-5.972615,2.33e-09,A
1,gru_baseline_1234,gru_optuna_1234,23.341496,0.00e+00,B
2,gru_baseline_271828183,gru_rs_271828183,33.392262,0.00e+00,B
3,gru_baseline_271828183,gru_optuna_271828183,35.350095,0.00e+00,B
4,gru_baseline_31415,gru_rs_31415,23.809281,0.00e+00,B
5,gru_baseline_31415,gru_optuna_31415,28.189534,0.00e+00,B
6,gru_baseline_42,gru_rs_42,-14.405832,0.00e+00,A
7,gru_baseline_42,gru_optuna_42,30.951242,0.00e+00,B
8,gru_baseline_73,gru_rs_73,15.984493,0.00e+00,B
9,gru_baseline_73,gru_optuna_73,-8.825714,0.00e+00,A




Significance Summary:
Significant tests (p < 0.05): 40/40
Percentage: 100.0%


In [20]:
# Final Summary
print("\n" + "="*80)
print("INTERPRETATION")
print("="*80 + "\n")

if len(dm_df) > 0:
    print("Legend:")
    print("  Model A: Baseline model")
    print("  Model B: RS or Optuna optimized model")
    print("  DM-Stat: Diebold-Mariano statistic (positive = B is better, negative = A is better)")
    print("  P-value: Significance level (< 0.05 indicates statistically significant difference)")
    print("  Better Model: A or B - which model performs better")
    print("\n" + "="*80)
    
    # Extract model names and determine comparison type
    sig_tests = dm_df[dm_df['P-value'] < 0.05]
    
    if len(sig_tests) > 0:
        print(f"\n✓ Found {len(sig_tests)} statistically significant pairwise comparisons (p < 0.05):\n")
        for idx, (_, row) in enumerate(sig_tests.iterrows(), 1):
            print(f"{idx}. {row['Model A']} vs {row['Model B']}")
            print(f"   DM Statistic: {row['DM-Stat']:10.6f} | P-value: {row['P-value']:.2e}")
            print(f"   Better Model: {row['Better Model']} (significantly better performance)\n")
    else:
        print("\nNo statistically significant differences found at α=0.05 level.\n")
    
    print("="*80)
    print("Analysis complete!")
    print("="*80)


INTERPRETATION

Legend:
  Model A: Baseline model
  Model B: RS or Optuna optimized model
  DM-Stat: Diebold-Mariano statistic (positive = B is better, negative = A is better)
  P-value: Significance level (< 0.05 indicates statistically significant difference)
  Better Model: A or B - which model performs better


✓ Found 40 statistically significant pairwise comparisons (p < 0.05):

1. gru_baseline_1234 vs gru_rs_1234
   DM Statistic:  -5.972615 | P-value: 2.33e-09
   Better Model: A (significantly better performance)

2. gru_baseline_1234 vs gru_optuna_1234
   DM Statistic:  23.341496 | P-value: 0.00e+00
   Better Model: B (significantly better performance)

3. gru_baseline_271828183 vs gru_rs_271828183
   DM Statistic:  33.392262 | P-value: 0.00e+00
   Better Model: B (significantly better performance)

4. gru_baseline_271828183 vs gru_optuna_271828183
   DM Statistic:  35.350095 | P-value: 0.00e+00
   Better Model: B (significantly better performance)

5. gru_baseline_31415 vs gr

In [21]:
# Direct Comparison: RS vs Optuna Models
print("\n" + "="*80)
print("DIRECT COMPARISON: RS vs OPTUNA (Same Model, Same Seed)")
print("="*80 + "\n")

# Create RS vs Optuna comparison pairs
rs_optuna_pairs = {}

for model in unique_models:
    for seed in unique_seeds:
        # Find RS column for this model-seed
        rs_key = (model, 'rs', seed)
        
        # Find Optuna column for this model-seed
        optuna_key = (model, 'optuna', seed)
        
        if rs_key in parsed_models and optuna_key in parsed_models:
            rs_col = parsed_models[rs_key]
            optuna_col = parsed_models[optuna_key]
            pair_name = f"{model}_{seed}_rs_vs_optuna"
            rs_optuna_pairs[pair_name] = (rs_col, optuna_col)

print(f"Total RS vs Optuna comparisons: {len(rs_optuna_pairs)}\n")

# Conduct RS vs Optuna DM tests
rs_optuna_results = []

for pair_name, (col_rs, col_optuna) in rs_optuna_pairs.items():
    # Get predictions
    y_rs = df[col_rs].values
    y_optuna = df[col_optuna].values
    
    # Find common period (where both predictions and actual values exist)
    mask = ~(np.isnan(y_actual) | np.isnan(y_rs) | np.isnan(y_optuna))
    
    if mask.sum() < 5:  # Need at least 5 observations
        print(f"\n{pair_name}: SKIPPED (insufficient data points)")
        continue
    
    # Extract aligned data
    y_true_aligned = y_actual[mask]
    y_rs_aligned = y_rs[mask]
    y_optuna_aligned = y_optuna[mask]
    
    # Conduct DM test
    try:
        result = diebold_mariano_test(y_true_aligned, y_rs_aligned, y_optuna_aligned)
        
        # Determine which model is better
        if result['dm_statistic'] > 0:
            better_model = 'Optuna'  # Optuna (model 2) is better
        else:
            better_model = 'RS'  # RS (model 1) is better
        
        rs_optuna_results.append({
            'Model RS': col_rs,
            'Model Optuna': col_optuna,
            'DM-Stat': result['dm_statistic'],
            'P-value': result['p_value'],
            'Better Model': better_model
        })
        
        print(f"{pair_name}: DM={result['dm_statistic']:8.4f}, p={result['p_value']:.2e}, Better={better_model}")
    
    except Exception as e:
        print(f"\n{pair_name}: ERROR - {str(e)}")

print(f"\n\nTotal RS vs Optuna tests conducted: {len(rs_optuna_results)}")

# Create results dataframe
rs_optuna_df = pd.DataFrame(rs_optuna_results)



DIRECT COMPARISON: RS vs OPTUNA (Same Model, Same Seed)

Total RS vs Optuna comparisons: 20

gru_1234_rs_vs_optuna: DM= 23.0189, p=0.00e+00, Better=Optuna
gru_271828183_rs_vs_optuna: DM= 29.3020, p=0.00e+00, Better=Optuna
gru_31415_rs_vs_optuna: DM= 27.7230, p=0.00e+00, Better=Optuna
gru_42_rs_vs_optuna: DM= 32.3364, p=0.00e+00, Better=Optuna
gru_73_rs_vs_optuna: DM=-15.6339, p=0.00e+00, Better=RS
lstm_1234_rs_vs_optuna: DM= 36.7842, p=0.00e+00, Better=Optuna
lstm_271828183_rs_vs_optuna: DM=  2.7002, p=6.93e-03, Better=Optuna
lstm_31415_rs_vs_optuna: DM=-24.1980, p=0.00e+00, Better=RS
lstm_42_rs_vs_optuna: DM= 14.9141, p=0.00e+00, Better=Optuna
lstm_73_rs_vs_optuna: DM=-20.2681, p=0.00e+00, Better=RS
tcn_1234_rs_vs_optuna: DM= 14.7031, p=0.00e+00, Better=Optuna
tcn_271828183_rs_vs_optuna: DM=-11.0085, p=0.00e+00, Better=RS
tcn_31415_rs_vs_optuna: DM= 27.2561, p=0.00e+00, Better=Optuna
tcn_42_rs_vs_optuna: DM= 42.0762, p=0.00e+00, Better=Optuna
tcn_73_rs_vs_optuna: DM=  7.6681, p=1.75e

In [22]:
# Display RS vs Optuna Results Table
print("\n" + "="*80)
print("RS vs OPTUNA COMPARISON RESULTS")
print("="*80 + "\n")

if len(rs_optuna_df) > 0:
    # Format p-values with E-notation
    rs_optuna_display = rs_optuna_df.copy()
    rs_optuna_display['P-value'] = rs_optuna_display['P-value'].apply(
        lambda x: f"{x:.2e}" if x < 0.0001 else f"{x:.6f}"
    )
    
    print(f"Total RS vs Optuna Comparisons: {len(rs_optuna_display)}\n")
    
    # Display the table
    display(rs_optuna_display)
    
    print("\n\nSignificance Summary:")
    print("="*80)
    sig_count = (rs_optuna_df['P-value'] < 0.05).sum()
    total_count = len(rs_optuna_df)
    print(f"Significant differences (p < 0.05): {sig_count}/{total_count}")
    print(f"Percentage: {100*sig_count/total_count:.1f}%")
    
    if sig_count > 0:
        # Count winners
        optuna_wins = (rs_optuna_df[rs_optuna_df['P-value'] < 0.05]['Better Model'] == 'Optuna').sum()
        rs_wins = (rs_optuna_df[rs_optuna_df['P-value'] < 0.05]['Better Model'] == 'RS').sum()
        print(f"\n  Optuna wins: {optuna_wins}")
        print(f"  RS wins: {rs_wins}")



RS vs OPTUNA COMPARISON RESULTS

Total RS vs Optuna Comparisons: 20



,Model RS,Model Optuna,DM-Stat,P-value,Better Model
0,gru_rs_1234,gru_optuna_1234,23.018910,0.00e+00,Optuna
1,gru_rs_271828183,gru_optuna_271828183,29.302046,0.00e+00,Optuna
2,gru_rs_31415,gru_optuna_31415,27.723045,0.00e+00,Optuna
3,gru_rs_42,gru_optuna_42,32.336410,0.00e+00,Optuna
4,gru_rs_73,gru_optuna_73,-15.633906,0.00e+00,RS
5,lstm_rs_1234,lstm_optuna_1234,36.784202,0.00e+00,Optuna
6,lstm_rs_271828183,lstm_optuna_271828183,2.700174,0.006930,Optuna
7,lstm_rs_31415,lstm_optuna_31415,-24.198031,0.00e+00,RS
8,lstm_rs_42,lstm_optuna_42,14.914060,0.00e+00,Optuna
9,lstm_rs_73,lstm_optuna_73,-20.268053,0.00e+00,RS




Significance Summary:
Significant differences (p < 0.05): 20/20
Percentage: 100.0%

  Optuna wins: 12
  RS wins: 8
